###STEP 1 — Read the CSV files from your Volume


In [0]:
customer_df = spark.read.csv("/Volumes/workspace/default/apachesparkguide/Customer_Updated.csv", header=True, inferSchema=True)
products_df = spark.read.csv("/Volumes/workspace/default/apachesparkguide/Products_Updated.csv", header=True, inferSchema=True)
transactions_df = spark.read.csv("/Volumes/workspace/default/apachesparkguide/Transaction_Updated.csv", header=True, inferSchema=True)

### Step 2 - Display if the extract step is complete

In [0]:
display(customer_df)
display(products_df)
display(transactions_df)

###STEP 3 — Create Bronze/Silver/Gold Folder Structure (Inside Volumes)

In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/default/apachesparkguide/bronze")
dbutils.fs.mkdirs("/Volumes/workspace/default/apachesparkguide/silver")
dbutils.fs.mkdirs("/Volumes/workspace/default/apachesparkguide/gold")


###STEP 4 — Write Raw Data to Bronze (as Delta)

In [0]:
customer_df.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/apachesparkguide/bronze/customer"
)

products_df.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/apachesparkguide/bronze/products"
)

transactions_df.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/apachesparkguide/bronze/transactions"
)


###STEP 5 — Read Bronze and Start Transformations (Silver Layer)

### Clean product names before joining 

In [0]:
bronze_customer = spark.read.format("delta").load(
    "/Volumes/workspace/default/apachesparkguide/bronze/customer"
)

bronze_products = spark.read.format("delta").load(
    "/Volumes/workspace/default/apachesparkguide/bronze/products"
)

bronze_transactions = spark.read.format("delta").load(
    "/Volumes/workspace/default/apachesparkguide/bronze/transactions"
)

In [0]:
from pyspark.sql import functions as F

# Clean + normalize transactions
bronze_transactions_clean = (
    bronze_transactions
        .withColumn("product_name", F.trim(F.lower("product_name")))
        .withColumn(
            "product_name",
            F.when(F.col("product_name").like("%iphone%"), "iphone")
             .when(F.col("product_name").like("%airpods%"), "airpods")
             .when(F.col("product_name").like("%macbook%"), "macbook")
             .when(F.col("product_name").like("%ipad%"), "ipad")
             .otherwise(F.col("product_name"))
        )
)

# Clean + normalize products
bronze_products_clean = (
    bronze_products
        .withColumn("product_name", F.trim(F.lower("product_name")))
        .withColumn(
            "product_name",
            F.when(F.col("product_name").like("%iphone%"), "iphone")
             .when(F.col("product_name").like("%airpods%"), "airpods")
             .when(F.col("product_name").like("%macbook%"), "macbook")
             .when(F.col("product_name").like("%ipad%"), "ipad")
             .otherwise(F.col("product_name"))
        )
)

bronze_transactions_clean.select("product_name").distinct().show(100, False)
bronze_products_clean.select("product_name").distinct().show(100, False)


In [0]:

silver_df = (
    bronze_transactions_clean
        .join(bronze_customer, "customer_id", "inner")
        .join(bronze_products_clean, "product_name", "inner")
)
display(silver_df)


####Write to Silver layer

In [0]:
silver_df.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/apachesparkguide/silver/combined"
)

###STEP 6 — Build the First Pipeline (AirPods After iPhone)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy("customer_id").orderBy("transaction_date")

df_with_lag = silver_df.withColumn(
    "prev_product",
    F.lag("product_name").over(w)
)

airpods_after_iphone = df_with_lag.filter(
    (F.col("product_name") == "airpods") &
    (F.col("prev_product") == "iphone")
)

####Write to Gold:

In [0]:
airpods_after_iphone.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/apachesparkguide/gold/airpods_after_iphone"
)

###STEP 7 — Validate Results

In [0]:
display(airpods_after_iphone)